In [20]:
import geopandas as gpd


#Load data
stops_PR = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\P+R_stations.gpkg")
stops_gtfs = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\stops_freq.gpkg")

#Reproject for analysis and mapping with accurate distances
stops_PR = stops_PR.to_crs("EPSG:32632")
stops_gtfs = stops_gtfs.to_crs("EPSG:32632")


#Spatially join the P+R stops to the GTFS stops
PR_to_gtfs = gpd.sjoin_nearest(
    stops_PR,
    stops_gtfs,
    how="left",
    distance_col="distance_m"
)

#We want a mapping so we only keep the relevant fields
PR_mapping = PR_to_gtfs[
    ["P+R_stop", "stop_id", "distance_m"]
]

#Filter out spatial joins which are too far away, around 200m
PR_mapping = PR_mapping[PR_mapping["distance_m"] < 200]

#Drop duplicates 
PR_mapping = PR_mapping.drop_duplicates(subset=["P+R_stop", "stop_id"])

PR_mapping.to_csv("pr_to_gtfs_mapping.csv", index=False)
